GD_Landsat_03_Select creates widget to pick images to use in animations.
Adds a column to LandsatMetadata.csv accordingly.
Some code from GD_Landsat_02, some from prompt: "in a python notebook, load a list of image file names into a pandas dataframe. Open each image in turn and have a checkbox labeled "Keep?" that stores true if checked or false if not checked into a new column of the dataframe"

Google AI says: you can use ipywidgets to create an interactive interface with an image display and a checkbox, and a pandas DataFrame to store the results. The process involves iterating through the images and using an observer function to update the DataFrame when the checkbox state changes. 
Then spits out some broken code, then fixes its code, and adds functionality...

TODO: Allow rename of old Keep column to add new Keep column(s).
TODO: how to make more flexible for new images (this code may be fine, but GD_Landsat_02 may need more work)
TODO: try again for keyboard shortcuts
TODO: transfer Keep column from neighboring glacier with similar satellite record and similar clouds
TODO: auto-save after 25 or so keeps

Current standards: 
"Keep" if the terminus of the glacier is clearly visible. 
Partial cloud allowed if there's no adjacent image close in time (~1 week). 
McBride had an image with georegistration errors - left that out. 
McBride is close to borders of path/row so get some apparent duplicates

See also: GD_Landsat_04_Animate, GD_Landsat_03a_SelectWithCloudFraction (on land or total)

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import glob

In [ ]:
#GOOD Create widget with image, Keep?, fwd, back, jump box but no Javascript (deleted other versions)
def label_images(df):
    index = 0
    
    # UI Elements
    filename_label = widgets.Label()
    img_widget = widgets.Image(width=300)
    checkbox = widgets.Checkbox(description='Keep?')
    
    btn_prev = widgets.Button(description=" ", icon="arrow-left") #description="Previous"
    btn_next = widgets.Button(description=" ", icon="arrow-right") #description="Next"
    
    # Jump Box: constraints values between 0 and the end of the dataframe
    jump_box = widgets.BoundedIntText(
        value=0, 
        min=0, 
        max=len(df)-1, 
        description='Jump:', #'Jump to Index:',
        style={'description_width': 'initial'}
    )
    
    nav_buttons = widgets.HBox([btn_prev, btn_next, jump_box])
    out = widgets.Output()

    def update_ui():
        nonlocal index
        with out:
            clear_output()
            if 0 <= index < len(df):
                fname = df.iloc[index]['ImageName']
                fpath = df.iloc[index]['ImagePath'] #AKB added
                filename_label.value = f"{fname} ({index}/{len(df)-1})" #f"File: {fname} | Index: {index} (Total: {len(df)})"
                
                # Update jump box value without triggering the observer
                jump_box.unobserve(on_jump, names='value')
                jump_box.value = index
                jump_box.observe(on_jump, names='value')
                
                with open(fpath, "rb") as file:
                    img_widget.value = file.read()
                
                # Cast to native bool for ipywidgets compatibility
                checkbox.value = bool(df.at[index, 'Keep?'])
            else:
                print("End of list.")

    def on_next_clicked(b):
        nonlocal index
        if index < len(df):
            df.at[index, 'Keep?'] = checkbox.value
            index += 1
            if index < len(df):
                update_ui()
            else:
                with out:
                    clear_output()
                    print("Finished! Final DataFrame:")
                    display(df)

    def on_prev_clicked(b):
        nonlocal index
        if index > 0:
            df.at[index, 'Keep?'] = checkbox.value
            index -= 1
            update_ui()

    def on_jump(change):
        nonlocal index
        # Save current state before jumping
        df.at[index, 'Keep?'] = checkbox.value
        index = change['new']
        update_ui()

    # Event Handlers
    btn_next.on_click(on_next_clicked)
    btn_prev.on_click(on_prev_clicked)
    jump_box.observe(on_jump, names='value')
    
    # Initial display
    update_ui()
    display(widgets.VBox([img_widget, filename_label, checkbox, nav_buttons, out]))

In [ ]:
# DEMO: Create a dummy image directory and some image files for demonstration
os.makedirs(os.path.join(folder_shp,"test_images"), exist_ok=True)
for i in range(8):
    img = Image.new('RGB', (100, 100), color = ('red' if i%2==0 else ('blue' if i%3==0 else 'green')))
    img.save(os.path.join(folder_shp,f'test_images/image_{i}.png'))

In [ ]:
# DEMO: Load file paths into a DataFrame
image_files = glob.glob(os.path.join(folder_shp,"test_images/*.png"))
df = pd.DataFrame(image_files, columns=['ImagePath'])
df['ImageName'] = df['ImagePath'].apply(os.path.basename)

# Initialize a new 'Keep?' column with False
#df['Keep?'] = False
# Force the column to native Python booleans during initialization to avoid "TraitError: The 'value' trait of a Checkbox instance expected a boolean, not the bool np.False_."
if 'Keep?' not in mdf.columns:
    df['Keep?'] = pd.Series([False] * len(df), dtype=object)

print("Initial DataFrame:")
display(df)

In [ ]:
# DEMO:
label_images(df)

In [ ]:
#Now for real:
folder_base = r'C:\Users\andyb\Documents\U\SEAN_Glacier-Dynamics' #os.path.join()
folder_shp = r'C:\Users\andyb\Documents\U\GEE-Courses\data'
file_path=os.path.join(folder_base,'glacierPropsLandsat.csv')

glaciers = pd.read_csv(file_path) #contains Name, LatCenter, LonCenter, two types of bounding boxes (see GD_Landsat_01_Setup).
glaciers['Name']

In [ ]:
#choose one glacier 
glacier = glaciers.iloc[12] #0=Margerie, 12=McBride
glacierdf=glaciers.iloc[[12]]
print('You chose: ' + glacier['Name'])
folder_out=os.path.join(folder_shp, glacier['Name'])
folder_fig=os.path.join(folder_out, 'Figures')

In [ ]:
# load metadata from CSV
file_meta = os.path.join(folder_out, 'LandsatMetadata.csv')
mdf = pd.read_csv(file_meta, parse_dates=['DATE_ACQUIRED', 'datetime'])
file_metaBackup = os.path.join(folder_out, 'LandsatMetadataBackup.csv')
mdf.to_csv(file_metaBackup, index=False)

print(f"Image info loaded from: {file_meta} and backed up to: {file_metaBackup}")

In [ ]:
#type(mdf) #pandas.core.frame.DataFrame
print(mdf.dtypes) #variable datetime has type datetime64[ns]
print(mdf.columns) #Index(['system:id', 'system:index', 'DATE_ACQUIRED', 'system:time_start', 'CLOUD_COVER', 'CLOUD_COVER_LAND', 'datetime', 'datetimestr'],      dtype='object')

In [ ]:
print(mdf['datetime'][0])
type(mdf['datetime'][0]) #pandas._libs.tslibs.timestamps.Timestamp #was str then I added parse_dates to the read_csv line

In [ ]:
mdf.iloc[0]

In [ ]:
# Initialize a new 'Keep?' column with False. Force the column to native Python booleans during initialization to avoid "TraitError: The 'value' trait of a Checkbox instance expected a boolean, not the bool np.False_."
if 'Keep?' not in mdf.columns:
    mdf['Keep?'] = pd.Series([False] * len(mdf), dtype=object)

#recreate columns (probably a better way to do this, or save in previous file, but hmmm...
if 'ImageName' not in mdf.columns:
    #mdf['ImageName'] = f'{mdf['system:id'].str[-8:]}_{mdf['system:id'].replace("/", "_")}.png') #add date first for better sort
    mdf['ImageName'] = mdf['system:id'].str[-8:] + '_' + mdf['system:id'].str.replace("/", "_") + '.png' #add date first for better sort
if 'ImagePath' not in mdf.columns:
    mdf['ImagePath'] = mdf['ImageName'].apply(lambda x: os.path.join(folder_out, x))

In [ ]:
label_images(mdf)

In [ ]:
# Save metadata to CSV
mdf.to_csv(file_meta, index=False)
print(f"Updated image info saved to: {file_meta}")

In [ ]:
# TODO ASIDE: Count points per year of True and False
mdf['year'] = mdf['datetime'].dt.year
points_per_year = mdf['year'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 6))
# Bar chart
bars = ax.bar(points_per_year.index, points_per_year.values,
              color='steelblue', edgecolor='black', alpha=0.8)
# Add value labels on top of bars
#for bar in bars:
#    height = bar.get_height()
#    ax.text(bar.get_x() + bar.get_width()/2, height + 0.5,
#            f'{int(height)}', ha='center', va='bottom', fontsize=10)

ax.set_xlabel('Year')
ax.set_ylabel('Images')
ax.grid(True, axis='y', linestyle='--', alpha=0.5)
ax.set_xticks(points_per_year.index) # Optional: force integer x-ticks
ax.tick_params(axis='x', rotation=90) #better(?) than plt.xticks(rotation=90)
#fig.savefig(os.path.join(folder_fig,glacier['Name']+'CountPerYear.png'), dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
#import seaborn as sns (not worth installing)
#sns.scatterplot(data=mdf, x='CLOUD_COVER', y='CLOUD_COVER_LAND', hue='Keep?')

#mdf.plot.scatter(x='CLOUD_COVER', y='CLOUD_COVER_LAND', c='Keep?', colormap='viridis')
fig, ax = plt.subplots(figsize=(6, 5))
plt.scatter(mdf['CLOUD_COVER'], mdf['CLOUD_COVER_LAND'], c=mdf['Keep?'], cmap='coolwarm')
plt.colorbar(label='Keep? (Red=True, Blue=False)')
ax.set_xlabel('Cloud cover')
ax.set_ylabel('Cloud cover over land')
#ax.set_title('Datetime vs Value (auto-formatted labels)')
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
fig.savefig(os.path.join(folder_fig,glacier['Name']+'CloudCoverLandVsTotalKeep.png'), dpi=300, bbox_inches='tight')
plt.show()